<a href="https://colab.research.google.com/github/TheVit808/CarteiraDeMarkowitzETFs/blob/main/interactive_portfolio_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dashboard Interativo de Otimização de Portfólio (Jupyter Notebook)

Este notebook oferece um dashboard interativo para análise e otimização de portfólio, utilizando ETFs diversificados. Ele permite ajustar parâmetros como a seleção de ativos, período de análise e taxa livre de risco, visualizando os resultados dinamicamente através de gráficos interativos.

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação das bibliotecas necessárias (se estiver rodando no Google Colab ou ambiente sem elas)
%pip install yfinance pandas numpy matplotlib plotly scipy ipywidgets --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.optimize import minimize
from IPython.display import display, HTML
import ipywidgets as widgets
from datetime import date

# Configurações de plotagem (para matplotlib, se usado)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

print('Bibliotecas importadas com sucesso!')

## 2. Seleção e Explicação dos Ativos (ETFs)

In [ ]:
etfs_dict = {
    "SPY": "SPDR S&P 500 ETF (Ações EUA - Amplo Mercado)",
    "XLK": "Technology Select Sector SPDR Fund (Ações EUA - Tecnologia)",
    "XLF": "Financial Select Sector SPDR Fund (Ações EUA - Financeiro)",
    "XLE": "Energy Select Sector SPDR Fund (Ações EUA - Energia)",
    "XLP": "Consumer Staples Select Sector SPDR Fund (Ações EUA - Consumo Essencial)",
    "EFA": "iShares MSCI EAFE ETF (Ações Internacionais - Mercados Desenvolvidos)",
    "TLT": "iShares 20+ Year Treasury Bond ETF (Renda Fixa - Títulos do Tesouro EUA)",
    "GLD": "SPDR Gold Shares (Commodities - Ouro)",
    "VNQ": "Vanguard Real Estate ETF (Setor Imobiliário - REITs EUA)",
    "BITO": "ProShares Bitcoin Strategy ETF (Criptomoedas - Futuros de Bitcoin)",
    "UUP": "Invesco DB US Dollar Index Bullish Fund (Moedas - Dólar Americano)"
}
etf_tickers = list(etfs_dict.keys())

display(HTML('<h3>ETFs Selecionados e Descrições:</h3>'))
for ticker, description in etfs_dict.items():
    display(HTML(f'- <b>{ticker}</b>: {description}'))

## 3. Funções Auxiliares para Otimização de Portfólio

In [ ]:
def portfolio_return(weights, annual_returns):
    return np.sum(annual_returns * weights)

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_sharpe(weights, annual_returns, cov_matrix, risk_free_rate):
    ret = portfolio_return(weights, annual_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return (ret - risk_free_rate) / vol if vol != 0 else 0

def negative_sharpe(weights, annual_returns, cov_matrix, risk_free_rate):
    return -portfolio_sharpe(weights, annual_returns, cov_matrix, risk_free_rate)

def get_portfolio_metrics(weights, annual_returns, cov_matrix, risk_free_rate):
    ret = portfolio_return(weights, annual_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return ret, vol, sharpe

## 4. Interface Interativa (IPyWidgets)

In [ ]:
# Widgets de controle
selected_etfs_widget = widgets.SelectMultiple(
    options=etf_tickers,
    value=etf_tickers,
    description='ETFs:',
    disabled=False,
    rows=len(etf_tickers),
    continuous_update=False
)

start_date_widget = widgets.DatePicker(
    description='Data de Início:',
    value=date(2015, 1, 1),
    disabled=False
)

end_date_widget = widgets.DatePicker(
    description='Data de Fim:',
    value=date.today(),
    disabled=False
)

risk_free_rate_widget = widgets.FloatSlider(
    value=2.0,
    min=0.0,
    max=10.0,
    step=0.1,
    description='Taxa Livre de Risco Anual (%):',
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Layout dos widgets
controls = widgets.VBox([selected_etfs_widget, start_date_widget, end_date_widget, risk_free_rate_widget])

output = widgets.Output()

def run_analysis(selected_etfs, start_date, end_date, risk_free_rate_percent):
    with output:
        output.clear_output(wait=True)
        if not selected_etfs:
            print("Por favor, selecione pelo menos um ETF para continuar.")
            return

        risk_free_rate = risk_free_rate_percent / 100

        # Download dos dados
        try:
            data = yf.download(list(selected_etfs), start=start_date, end=end_date)['Close']
            data = data.dropna()
            if data.empty:
                print("Não foi possível baixar dados para os ETFs selecionados no período especificado. Tente ajustar as datas ou os ETFs.")
                return
        except Exception as e:
            print(f"Erro ao baixar dados: {e}. Verifique os tickers e o período.")
            return

        # Cálculo dos retornos e métricas básicas
        returns = data.pct_change().dropna()
        if returns.empty:
            print("Dados insuficientes para calcular retornos. Ajuste o período.")
            return

        annual_returns = returns.mean() * 252
        annual_volatility = returns.std() * np.sqrt(252)

        display(HTML('<h3>Dados Históricos de Fechamento (Primeiras 5 linhas):</h3>'))
        display(data.head())

        display(HTML('<h3>Métricas Anuais (Retorno e Volatilidade):</h3>'))
        metrics_df = pd.DataFrame({
            'Retorno Anual': annual_returns,
            'Volatilidade Anual': annual_volatility
        }).sort_values(by='Retorno Anual', ascending=False)
        display(metrics_df)

        # --- Análise de Correlação Móvel ---
        display(HTML('<h3>Correlação Móvel (Janela de 1 Ano):</h3>'))
        if len(selected_etfs) >= 2:
            asset1_corr = selected_etfs[0]
            asset2_corr = selected_etfs[1] if len(selected_etfs) > 1 else selected_etfs[0]

            if asset1_corr != asset2_corr:
                rolling_corr = returns[asset1_corr].rolling(window=252).corr(returns[asset2_corr])
                fig_corr = go.Figure()
                fig_corr.add_trace(go.Scatter(x=rolling_corr.index, y=rolling_corr, mode='lines', name=f'{asset1_corr} vs {asset2_corr}'))
                fig_corr.update_layout(title=f'Correlação Móvel: {asset1_corr} vs {asset2_corr}',
                                       xaxis_title='Data', yaxis_title='Correlação')
                fig_corr.show()
            else:
                print("Selecione pelo menos dois ativos diferentes para visualizar a correlação móvel.")
        else:
            print("Selecione pelo menos dois ETFs para visualizar a correlação móvel.")

        # --- Índice Sharpe Individual e Temporal ---
        display(HTML('<h3>Índice Sharpe:</h3>'))
        sharpe_ratios = (annual_returns - risk_free_rate) / annual_volatility

        display(HTML('<h4>Índice Sharpe por Ativo</h4>'))
        fig_sharpe_bar = go.Figure(data=[go.Bar(x=sharpe_ratios.sort_values(ascending=False).index, y=sharpe_ratios.sort_values(ascending=False))])
        fig_sharpe_bar.update_layout(title='Índice Sharpe por Ativo',
                                     xaxis_title='Ativo', yaxis_title='Índice Sharpe')
        fig_sharpe_bar.show()

        display(HTML('<h4>Índice Sharpe Individual ao Longo do Tempo (Janela de 1 Ano)</h4>'))
        rolling_sharpe = (returns.rolling(window=252).mean() * 252 - risk_free_rate) / (returns.rolling(window=252).std() * np.sqrt(252))

        fig_rolling_sharpe = go.Figure()
        for col in rolling_sharpe.columns:
            fig_rolling_sharpe.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe[col], mode='lines', name=col))
        fig_rolling_sharpe.update_layout(title='Índice Sharpe Individual ao Longo do Tempo',
                                         xaxis_title='Data', yaxis_title='Índice Sharpe')
        fig_rolling_sharpe.show()

        # --- Gráfico de Risco e Retorno com Ativo Livre de Risco ---
        display(HTML('<h3>Risco vs. Retorno dos Ativos com Linha de Ativo Livre de Risco:</h3>'))

        fig_risk_return_rf = go.Figure()
        fig_risk_return_rf.add_trace(go.Scatter(
            x=annual_volatility, y=annual_returns,
            mode='markers+text', text=annual_returns.index, textposition='top center',
            marker=dict(size=10, color=sharpe_ratios, colorscale='Viridis', showscale=True, colorbar=dict(title='Índice Sharpe')),
            name='Ativos Individuais',
            hoverinfo='text',
            hovertext=[f'Ativo: {asset}<br>Retorno: {ret:.2%}<br>Volatilidade: {vol:.2%}<br>Sharpe: {sharpe:.2f}'
                       for asset, ret, vol, sharpe in zip(annual_returns.index, annual_returns, annual_volatility, sharpe_ratios)]
        ))

        if not sharpe_ratios.empty:
            max_sharpe_asset = sharpe_ratios.idxmax()
            max_sharpe_return = annual_returns[max_sharpe_asset]
            max_sharpe_volatility = annual_volatility[max_sharpe_asset]

            x_cal = np.linspace(0, max_sharpe_volatility * 1.5, 100)
            y_cal = risk_free_rate + (max_sharpe_return - risk_free_rate) / max_sharpe_volatility * x_cal
            fig_risk_return_rf.add_trace(go.Scatter(x=x_cal, y=y_cal, mode='lines', name='Linha de Alocação de Capital (Simplificada)',
                                                    line=dict(dash='dash', color='red')))

        fig_risk_return_rf.update_layout(title='Risco vs. Retorno dos Ativos com Linha de Ativo Livre de Risco',
                                         xaxis_title='Volatilidade Anual (Desvio Padrão)',
                                         yaxis_title='Retorno Anual Esperado')
        fig_risk_return_rf.show()

        # --- Otimização de Portfólio (Markowitz) ---
        display(HTML('<h3>Otimização de Portfólio (Markowitz):</h3>'))
        if len(selected_etfs) >= 2:
            cov_matrix_annual = returns.cov() * 252
            num_assets = len(selected_etfs)
            num_portfolios = 10000

            results = np.zeros((3, num_portfolios))
            weights_record = []

            for i in range(num_portfolios):
                weights = np.random.random(num_assets)
                weights /= np.sum(weights)
                weights_record.append(weights)

                p_return, p_volatility, p_sharpe = get_portfolio_metrics(weights, annual_returns, cov_matrix_annual, risk_free_rate)

                results[0, i] = p_volatility
                results[1, i] = p_return
                results[2, i] = p_sharpe

            # Encontrar a carteira com o maior Sharpe Ratio
            max_sharpe_idx = np.argmax(results[2])
            optimal_portfolio_volatility = results[0, max_sharpe_idx]
            optimal_portfolio_return = results[1, max_sharpe_idx]
            optimal_portfolio_sharpe = results[2, max_sharpe_idx]
            optimal_weights = weights_record[max_sharpe_idx]

            # Encontrar a carteira de mínima variância
            min_vol_idx = np.argmin(results[0])
            min_vol_portfolio_volatility = results[0, min_vol_idx]
            min_vol_portfolio_return = results[1, min_vol_idx]
            min_vol_weights = weights_record[min_vol_idx]

            display(HTML('<h4>Carteira Ótima (Maior Sharpe Ratio):</h4>'))
            display(HTML(f' Retorno: <b>{optimal_portfolio_return:.2%}</b>'))
            display(HTML(f' Volatilidade: <b>{optimal_portfolio_volatility:.2%}</b>'))
            display(HTML(f' Sharpe Ratio: <b>{optimal_portfolio_sharpe:.2f}</b>'))
            display(HTML(' Pesos:'))
            optimal_weights_df = pd.DataFrame({
                'Ativo': list(selected_etfs),
                'Peso': [f'{w:.2%}' for w in optimal_weights]
            })
            display(optimal_weights_df.set_index('Ativo'))

            display(HTML('<h4>Carteira de Mínima Variância:</h4>'))
            display(HTML(f' Retorno: <b>{min_vol_portfolio_return:.2%}</b>'))
            display(HTML(f' Volatilidade: <b>{min_vol_portfolio_volatility:.2%}</b>'))
            display(HTML(' Pesos:'))
            min_vol_weights_df = pd.DataFrame({
                'Ativo': list(selected_etfs),
                'Peso': [f'{w:.2%}' for w in min_vol_weights]
            })
            display(min_vol_weights_df.set_index('Ativo'))

            # --- Gráfico de Risco e Retorno (Fronteira Eficiente, Carteira Ótima e Mínima Variância) ---
            display(HTML('<h3>Fronteira Eficiente, Carteira Ótima e Mínima Variância:</h3>'))

            fig_frontier = go.Figure()
            fig_frontier.add_trace(go.Scatter(
                x=results[0,:], y=results[1,:], mode='markers',
                marker=dict(size=5, color=results[2,:], colorscale='Viridis', showscale=True, colorbar=dict(title='Sharpe Ratio')),
                name='Portfólios Simulados',
                hoverinfo='text',
                hovertext=[f'Volatilidade: {vol:.2%}<br>Retorno: {ret:.2%}<br>Sharpe: {sharpe:.2f}'
                           for vol, ret, sharpe in zip(results[0,:], results[1,:], results[2,:])]
            ))

            fig_frontier.add_trace(go.Scatter(
                x=[min_vol_portfolio_volatility], y=[min_vol_portfolio_return],
                mode='markers', marker=dict(size=15, color='red', symbol='star'),
                name='Carteira de Mínima Variância',
                hoverinfo='text',
                hovertext=f'Carteira de Mínima Variância<br>Retorno: {min_vol_portfolio_return:.2%}<br>Volatilidade: {min_vol_portfolio_volatility:.2%}'
            ))

            fig_frontier.add_trace(go.Scatter(
                x=[optimal_portfolio_volatility], y=[optimal_portfolio_return],
                mode='markers', marker=dict(size=15, color='green', symbol='star'),
                name='Carteira Ótima (Maior Sharpe)'
            ))

            fig_frontier.add_trace(go.Scatter(
                x=annual_volatility, y=annual_returns,
                mode='markers+text', text=annual_returns.index, textposition='top center',
                marker=dict(size=10, color='black'),
                name='Ativos Individuais',
                hoverinfo='text',
                hovertext=[f'Ativo: {asset}<br>Retorno: {ret:.2%}<br>Volatilidade: {vol:.2%}'
                           for asset, ret, vol in zip(annual_returns.index, annual_returns, annual_volatility)]
            ))

            fig_frontier.update_layout(title='Fronteira Eficiente, Carteira Ótima e Mínima Variância',
                                       xaxis_title='Volatilidade Anual (Desvio Padrão)',
                                       yaxis_title='Retorno Anual Esperado')
            fig_frontier.show()

        else:
            print("Selecione pelo menos dois ETFs para realizar a otimização de portfólio.")

display(widgets.HBox([controls, output]))
widgets.interactive_output(run_analysis, {
    'selected_etfs': selected_etfs_widget,
    'start_date': start_date_widget,
    'end_date': end_date_widget,
    'risk_free_rate_percent': risk_free_rate_widget
})

## 5. Conclusão

Este dashboard interativo demonstra a aplicação de conceitos de finanças quantitativas para análise e otimização de portfólios. As ferramentas de visualização e os controles interativos permitem uma exploração flexível dos dados e dos resultados da otimização de Markowitz.